# SentinelRAG — 04: Retrieval & Reranking Experiments

This notebook benchmarks and proves the retrieval architecture decisions:
1. **Dense Vector Search (Qdrant + nomic-embed-text)**: Semantic semantic matching.
2. **Lexical BM25 Search**: Exact keyword, numeric threshold (e.g. `₹3,000`), and acronym (`MFA`, `CFO`, `PL`) matching.
3. **Reciprocal Rank Fusion (RRF)**: Combining the best of both approaches.
4. **Reranker Scoring**: Precision ranking for LLM context curation.

In [1]:
import sys
from pathlib import Path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from rich import print as rprint
from rich.table import Table
from src.ingestion.chunking import load_chunks_from_json
from src.retrieval.vector_store import SentinelVectorStore
from src.retrieval.bm25_retriever import SentinelBM25Retriever
from src.retrieval.hybrid import reciprocal_rank_fusion
from src.retrieval.reranker import SentinelReranker
from config.settings import settings

# Load cached chunk store
chunks_path = settings.processed_data_dir / "chunks.json"
chunks = load_chunks_from_json(chunks_path)
print(f"Loaded {len(chunks)} indexed chunks from {chunks_path}")

Loaded 26 indexed chunks from C:\Users\Keshav\RAG\SentinelRAG\data\processed\chunks.json


## 1. Test Query Comparison: Vector vs BM25
We test with a query requiring both numeric and semantic sensitivity:
`"What is the maximum hotel accommodation claim limit?"`

In [2]:
query = "What is the maximum hotel accommodation claim limit?"

vector_store = SentinelVectorStore()
bm25 = SentinelBM25Retriever(chunks)

v_results = vector_store.search(query, top_k=5)
b_results = bm25.search(query, top_k=5)

table = Table(title=f"Search Comparison: '{query}'")
table.add_column("Rank", justify="center")
table.add_column("Vector Top Result (Cosine)", style="cyan")
table.add_column("BM25 Top Result (Lexical)", style="green")

for r in range(min(len(v_results), len(b_results))):
    table.add_row(
        str(r + 1),
        f"{v_results[r][0].chunk_id} ({v_results[r][1]:.3f})",
        f"{b_results[r][0].chunk_id} ({b_results[r][1]:.2f})"
    )

rprint(table)

Search Comparison: 'What is the maximum hotel accommodation claim
                             limit?'                             
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Vector Top Result (Cosine) ┃ BM25 Top Result (Lexical) ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
└──────┴────────────────────────────┴───────────────────────────┘

## 2. Reciprocal Rank Fusion (RRF)
We fuse the dense and lexical ranked candidate lists using RRF ($k=60$).

In [3]:
fused = reciprocal_rank_fusion(v_results, b_results, k=60, top_n=5)

rrf_table = Table(title="Reciprocal Rank Fusion Results")
rrf_table.add_column("RRF Rank", justify="center")
rrf_table.add_column("Chunk ID", style="yellow")
rrf_table.add_column("RRF Score", justify="right")
rrf_table.add_column("Vector Rank", justify="center")
rrf_table.add_column("BM25 Rank", justify="center")

for rank, (chunk, score, info) in enumerate(fused):
    rrf_table.add_row(
        str(rank + 1),
        chunk.chunk_id,
        str(score),
        str(info.get("vector_rank", "-")),
        str(info.get("bm25_rank", "-"))
    )

rprint(rrf_table)

                    Reciprocal Rank Fusion Results                     
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ RRF Rank ┃ Chunk ID           ┃ RRF Score ┃ Vector Rank ┃ BM25 Rank ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│    1     │ Travel_Policy_c004 │   0.01639 │    None     │     1     │
│    2     │ Leave_Policy_c006  │   0.01613 │    None     │     2     │
│    3     │ Travel_Policy_c010 │   0.01587 │    None     │     3     │
│    4     │ Travel_Policy_c003 │   0.01562 │    None     │     4     │
│    5     │ Travel_Policy_c005 │   0.01538 │    None     │     5     │
└──────────┴────────────────────┴───────────┴─────────────┴───────────┘